Лабораторная 2. Реализация сверточной нейронной сети для классификации цифр

Исходные данные - [MNIST](http://yann.lecun.com/exdb/mnist/). Обучающая выборка - 70%, тестовая - 30%.

Количество слоев, размерность свертки и гиперпараметры свертки и пулинга на своё усмотрение.

Обучение - обратное распространение ошибки, средняя ошибка - 0,0001.

Реализовано:
- вычисление матрицы неточностей, по которой должны вычисляться показатели accuracy, precision, recall, F-мера, строится ROC-кривая, вычислятся AUC.
- t-SNE и оценка причин ошибочной классификации.

# Import

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc
from category_encoders import TargetEncoder

from pprint import pprint
from dotenv import load_dotenv

In [2]:
load_dotenv()
PATH_DATA = Path(os.getenv('PATH_DATA'))
# sklearn.set_config(transform_output="pandas")
sklearn.set_config(transform_output="default")


from src.layers import ConvLayer, LinearLayer
from src.activations import ReLUActivation, SigmoidActivation
from src.structure_layers import FlattenLayer
from src.losses import CrossEntropyLoss
from src.data_loaders import ShuffleLoader, datasets
from src.optimizers import GDOptimizer
from src.models import CNN

# Данные

## Загружаем данные

In [13]:
X_train, X_val, X_test, y_train, y_val, y_test = datasets.load_mnist_dataset(save_path=PATH_DATA, val_size=0.2)

In [14]:
model = CNN([
    ConvLayer(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=0, bias=True, padding_mode='constant'), ReLUActivation(),
    ConvLayer(in_channels=32, out_channels=16, kernel_size=3, stride=1, padding=0, bias=True, padding_mode='constant'), ReLUActivation(),
    ConvLayer(in_channels=16, out_channels=8, kernel_size=3, stride=1, padding=0, bias=True, padding_mode='constant'), ReLUActivation(),
    FlattenLayer(),
    LinearLayer(in_features=8*22*22, out_features=10, bias=True), SigmoidActivation()
])

# print(model(X_test[:1]).shape)

# Инициализируем Loss и оптимизатор
loss = CrossEntropyLoss()
data_loader = ShuffleLoader(batch_size=16)
optimizer = GDOptimizer(
    model_weights_layers=model.get_weights_layers(),
    data_loader=data_loader,
    lr=0.001
)

In [15]:
def postprocess(y_probs: np.ndarray) -> np.ndarray:
    y_preds = np.zeros_like(y_probs, dtype=int)
    max_indices = np.argmax(y_probs, axis=1)
    y_preds[np.arange(y_probs.shape[0]), max_indices] = 1
    return y_preds

In [ ]:
# Задаём параметры обучения и запускаем его
n_epochs = 5
verbose_n_batch_multiple = 1
model.train_model(
    n_epochs,
    X_train, y_train, X_val, y_val,
    loss, optimizer,
    postprocess=postprocess, count_metric=accuracy_score,
    verbose_n_batch_multiple=verbose_n_batch_multiple, verbose_statistic='EMA'
)